# A Petrinaut net, verified and controlled

HASH's [Petrinaut](https://github.com/hashintel/hash/tree/main/libs/%40hashintel/petrinaut)
edits Petri nets and exports them as YAML. `zrth.petri.load_sdcpn` turns such an export into a
reactive module: places become integer state variables, transitions guarded updates, and *which*
transition fires a nondeterministic external input. From there the usual toolbox applies. This
notebook takes the **L0 cyclic-resource control** instance (`minimal_manufacturing_cell.yaml`,
with its problem statement in the PDF next to it) and

1. lets Z3 find the deadlock the problem statement describes,
2. verifies the greedy baseline dispatcher and watches it fail,
3. trains a dispatch policy with REINFORCE against the module, and
4. verifies the learned policy under **every** interleaving of the plant's moves.

## The net

Three jobs, three machines, each job needs two machines in a cycle (A→B, B→C, C→A) and holds the
first until it finishes on the second. Dispatching a job is a **controllable** transition (the
controller's lever); completing one is **uncontrollable** (the plant does it when the second
machine is free). If all three jobs are dispatched before any completes, each holds the machine
the next one needs: a circular wait, and nothing can ever fire again.

In [ ]:
from pathlib import Path

import torch

from zrth import X
from zrth.eval import execute_init, execute_update
from zrth.petri import load_sdcpn

HERE = next(p for p in (Path("."), Path("tutorials/petrinaut")) if (p / "minimal_manufacturing_cell.yaml").exists())
net = load_sdcpn(HERE / "minimal_manufacturing_cell.yaml")

places = list(net.places)
idx = {name: i for i, name in enumerate(places)}
CTRL = [i for i, t in enumerate(net.transitions) if net.control[t] == "controllable"]
UNCTRL = [i for i, t in enumerate(net.transitions) if net.control[t] == "uncontrollable"]
DONE = ["CompletedAB", "CompletedBC", "CompletedCA"]

print(f"{len(places)} places, {len(net.transitions)} transitions; initial marking:",
      {p: c for p, c in net.marking.items() if c})
for i, t in enumerate(net.transitions):
    print(f"  {i}  {net.control[t]:14s} {t}")

12 places, 6 transitions; initial marking: {'OrderAB': 1, 'OrderBC': 1, 'OrderCA': 1, 'MachineAFree': 1, 'MachineBFree': 1, 'MachineCFree': 1}
  0  controllable   Start A-to-B job
  1  uncontrollable Complete A-to-B job
  2  controllable   Start B-to-C job
  3  uncontrollable Complete B-to-C job
  4  controllable   Start C-to-A job
  5  uncontrollable Complete C-to-A job


## Stepping the module

A marking is a tuple of token counts in place order. `fire(m, i)` runs the module's update block
with the external input set to transition `i`; a disabled choice leaves the marking unchanged.
Enabledness is read off the arcs the importer exposes (`net.pre`), and the run is over when
every job sits in its completed place.

In [ ]:
def reset():
    state = {}
    execute_init(state, net.module.atoms)
    return tuple(int(state[X(v)].item()) for v in net.places.values())


def fire(m, i):
    state = {v: torch.tensor([[c]]) for v, c in zip(net.places.values(), m)}
    state[net.fire] = torch.tensor([[i]])
    execute_update(state, net.module.atoms)
    return tuple(int(state[X(v)].item()) for v in net.places.values())


def enabled(m):
    return [i for i, pre in enumerate(net.pre) if all(m[idx[p]] >= w for p, w in pre.items())]


def completed(m):
    return all(m[idx[p]] >= 1 for p in DONE)


def show(m):
    return {p: c for p, c in zip(places, m) if c}


m = reset()
for t in ["Start A-to-B job", "Complete A-to-B job"]:
    m = fire(m, net.transitions.index(t))
print("after dispatching and completing A-to-B:", show(m))

after dispatching and completing A-to-B: {'OrderBC': 1, 'OrderCA': 1, 'MachineAFree': 1, 'MachineBFree': 1, 'MachineCFree': 1, 'CompletedAB': 1}


## Z3 finds the deadlock

Unroll the module's transition relation for `k` rounds with the firing choice symbolic, so Z3
searches over **all** interleavings at once, and ask for a marking where nothing is enabled but
the work is not done. The witness is the unsafe wave from the problem statement: three
dispatches, no completion.

In [ ]:
import numpy as np
import z3
from zrth import z3 as zz3


def deadlock_witness(steps):
    q = [[z3.Int(f"{p}_{k}") for p in places] for k in range(steps + 1)]
    f = [z3.Int(f"fire_{k}") for k in range(steps)]
    s = z3.Solver()
    s.add(*[q[0][idx[p]] == c for p, c in net.marking.items()])
    for k in range(steps):
        s.add(f[k] >= 0, f[k] < len(net.transitions))
        st = {v: np.array([[q[k][i]]]) for i, v in enumerate(net.places.values())}
        st[net.fire] = np.array([[f[k]]])
        for atom in net.module.atoms:
            for term in atom.update:
                st.update(zip(term.write, zz3.eval(term.itype, [st[w] for w in term.read])))
        s.add(*[q[k + 1][i] == st[X(v)].item() for i, v in enumerate(net.places.values())])
    last = q[steps]
    nothing_enabled = z3.And(*[z3.Not(z3.And(*[last[idx[p]] >= w for p, w in pre.items()])) for pre in net.pre])
    not_done = z3.Not(z3.And(*[last[idx[p]] >= 1 for p in DONE]))
    s.add(nothing_enabled, not_done)
    if s.check() != z3.sat:
        return None
    model = s.model()
    return [net.transitions[model.eval(f[k]).as_long()] for k in range(steps)]


for k in range(1, 6):
    witness = deadlock_witness(k)
    print(f"deadlock within {k} steps:", " -> ".join(witness) if witness else "none")
    if witness:
        break

deadlock within 1 steps: none
deadlock within 2 steps: none
deadlock within 3 steps: Start A-to-B job -> Start C-to-A job -> Start B-to-C job


## Verifying a dispatch policy under every interleaving

A dispatch policy maps a marking to a controllable transition, or to *wait*. The closed loop is
the net with the controller's move added to the plant's: from every marking, the successors are
the policy's dispatch (if any) **and** every enabled completion, so every interleaving of
controller and plant is explored. The state space is finite and small, so the reachable set is
enumerated exactly. The two specifications of the problem statement become:

- **φ1, deadlock freedom**: no reachable marking is stuck with work left;
- **φ2, completion**: every maximal run ends with all jobs completed (a policy that waits
  forever also fails here, so safety by blocking does not count).

The baseline, dispatching whenever a job's first machine is free, is exactly the greedy policy
the problem statement warns about.

In [ ]:
def verify(choose):
    "choose(m) -> controllable transition to fire, or None to wait. Returns (ok, states, counterexample)."
    start = reset()
    parent, frontier, seen = {start: None}, [start], {start}
    while frontier:
        m = frontier.pop()
        moves = [t for t in UNCTRL if t in enabled(m)]
        if (a := choose(m)) is not None:
            moves.append(a)
        if not moves and not completed(m):
            trace, cur = [], m
            while parent[cur] is not None:
                cur, t = parent[cur]
                trace.append(net.transitions[t])
            return False, len(seen), list(reversed(trace))
        for t in moves:
            n = fire(m, t)
            if n not in seen:
                seen.add(n)
                parent[n] = (m, t)
                frontier.append(n)
    return True, len(seen), None


def report(name, result):
    ok, states, trace = result
    print(f"{name}: {'VERIFIED' if ok else 'COUNTEREXAMPLE'} ({states} markings explored)")
    if trace:
        print("   ", " -> ".join(trace))


def greedy(m):
    ready = [t for t in CTRL if t in enabled(m)]
    return ready[0] if ready else None


report("greedy dispatcher", verify(greedy))

greedy dispatcher: COUNTEREXAMPLE (6 markings explored)
    Start A-to-B job -> Start B-to-C job -> Start C-to-A job


## Training a dispatcher

The controller is a small network over the marking with one output per controllable transition
plus *wait*; disabled dispatches are masked out. Each training step the agent moves, then the
plant completes a random enabled job with probability ½, so the dangerous orderings (three
dispatches before any completion) occur often enough to be learned from. Rewards: +1 per
completion, −5 for a deadlock, a small cost per step so that never dispatching does not pay.

One detail matters here: the returns are **not** normalised per episode. With sparse rewards the
return-to-go shrinks towards the end of a successful episode, so normalising pushes down exactly
the last dispatch of every run, and the policy learns to wait instead of finishing the job.

In [ ]:
import random
import torch.nn as nn
from zrth.torch import Module as TorchModule

WAIT = len(CTRL)
P_COMPLETE = 0.5


class PolicyNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(len(places), 16)
        self.fc2 = nn.Linear(16, len(CTRL) + 1)

    def forward(self, m):
        return self.fc2(torch.relu(self.fc1(m)))


def logits(m):
    out = policy(torch.tensor([m], dtype=torch.float32)).squeeze(0)
    ready = enabled(m)
    mask = torch.tensor([t in ready for t in CTRL] + [True])
    return out.masked_fill(~mask, float("-inf"))


def env_step(m, a, rng):
    if a < WAIT:
        m = fire(m, CTRL[a])
    reward = -0.05
    ready = [t for t in UNCTRL if t in enabled(m)]
    if ready and rng.random() < P_COMPLETE:
        m = fire(m, rng.choice(ready))
        reward += 1.0
    if completed(m):
        return m, reward, True
    if not enabled(m):
        return m, reward - 5.0, True
    return m, reward, False


torch.manual_seed(0)
rng = random.Random(0)
policy = TorchModule(PolicyNN(), combinatorial=True)
opt = torch.optim.Adam(policy.parameters(), lr=0.01)

for ep in range(2500):
    m, logps, rewards = reset(), [], []
    for _ in range(30):
        dist = torch.distributions.Categorical(logits=logits(m))
        a = dist.sample()
        logps.append(dist.log_prob(a))
        m, r, done = env_step(m, a.item(), rng)
        rewards.append(r)
        if done:
            break
    returns, G = [], 0.0
    for r in reversed(rewards):
        G = r + 0.99 * G
        returns.append(G)
    loss = -(torch.stack(logps) * torch.tensor(list(reversed(returns)))).sum()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if ep % 500 == 0 or ep == 2499:
        outcome = "completed" if completed(m) else "deadlock" if not enabled(m) else "cut off"
        print(f"episode {ep:4d}: {outcome} after {len(rewards):2d} steps")

episode    0: completed after  7 steps


episode  500: completed after  5 steps


episode 1000: completed after  7 steps


episode 1500: completed after  7 steps


episode 2000: completed after  5 steps


episode 2499: completed after  8 steps


## Verifying the learned dispatcher

Training saw random interleavings; verification explores all of them. The greedy (argmax)
policy is what would be deployed, so that is what gets verified.

In [ ]:
def learned(m):
    a = int(torch.argmax(logits(m)))
    return None if a == WAIT else CTRL[a]


report("learned dispatcher", verify(learned))

m = reset()
print("\ndecisions along one run:")
while not completed(m):
    a = learned(m)
    print(f"  holding {sum(m[idx[p]] for p in places if 'Holding' in p)} job(s) ->",
          net.transitions[a] if a is not None else "wait")
    m = fire(m, a) if a is not None else fire(m, next(t for t in UNCTRL if t in enabled(m)))

learned dispatcher: VERIFIED (9 markings explored)

decisions along one run:
  holding 0 job(s) -> Start A-to-B job
  holding 1 job(s) -> Start B-to-C job
  holding 2 job(s) -> wait
  holding 1 job(s) -> wait
  holding 0 job(s) -> Start C-to-A job
  holding 1 job(s) -> wait


## The larger instance

The second L0 instance, a flexible manufacturing cell with six product families, five machines,
alternative routes and a three-job release wave, loads the same way. Its problem statement
quotes exact figures (1,009 reachable markings with three dispatch credits, 1,209 deadlocked
markings in the balanced scenario) that a reachability analysis on this module can be checked
against; its `dispatch_credits` scenario is written as code rather than per place, which the
importer does not read yet.

In [ ]:
flexible = load_sdcpn(HERE / "flexible_manufacturing_cell_v2.yaml", scenario="Balanced workload")
print(f"{len(flexible.places)} places, {len(flexible.transitions)} transitions,",
      f"{sum(1 for c in flexible.control.values() if c == 'controllable')} controllable;",
      f"{sum(len(a.update) for a in flexible.module.atoms)} update terms")

29 places, 19 transitions, 9 controllable; 481 update terms


## Where this goes

One imported object did all the work: Z3 read its transition relation to find the deadlock, the
training loop stepped it as the environment, and the verifier enumerated it under the learned
controller. The step from here to the flexible cell is the step from a teaching example to the
benchmark: the same verifier, a policy over 29 places, and the problem statement's own numbers
to reproduce.